#  Generative Adversarial Networks (GAN)
## Introduction

In this tutorial, you will implement a GAN  using PyTorch on MNIST dataset. You will observe its failures and fix it through informed design choices.

## 1. Setup

First, import the required libraries:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else  "mps" if torch.backends.mps.is_available() else "cpu")
print(device)
torch.manual_seed(42)

## 2. Dataset and DataLoader

We will use the MNIST dataset for training.

In [ ]:


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

indices = (dataset.targets == 5) | (dataset.targets == 3) # if you want to keep images with the label 5
dataset.data, dataset.targets = dataset.data[indices], dataset.targets[indices]

batch_size = 512
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

images, _ = next(iter(loader))

plt.figure(figsize=(6,6))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.imshow(images[i][0], cmap="gray")
    plt.axis("off")
plt.suptitle("Real MNIST Samples")
plt.show()


## 3. Define the Generator
The generator takes a random noise vector and generates an image.

📌 Noise input shape: (batch_size, latent_size)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_size=64, hidden_size=256, image_size=28*28):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, image_size),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


## 4. Define the Discriminator
The discriminator classifies images as real or fake.

In [ ]:
class Discriminator(nn.Module):

    def __init__(self, hidden_size=256, image_size=28*28):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


## 5. Training loop
The training process involves updating both the generator and discriminator.

In [ ]:
def train_gan(epochs, loader, D, G, optimizer_D, optimizer_G, criterion, latent_size=64):
    losses_D = []
    losses_G = []

    for epoch in range(epochs):
        for real, _ in loader:

            batch_size = real.size(0)

            real = real.view(batch_size, -1).to(device)
            
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)
            
            # ---- Train Discriminator ----
            D.zero_grad()

            D_real = D(real)
            errD_real = criterion(D_real, real_labels)

            z = torch.randn(batch_size, latent_size).to(device)
            fake = G(z)
            D_fake = D(fake.detach())
            errD_fake = criterion(D_fake, fake_labels)
       
            loss_D = errD_real + errD_fake
            loss_D.backward()
            optimizer_D.step()

            # ---- Train Generator ----
            G.zero_grad()
            D_fake = D(fake)
            loss_G = criterion(D_fake, real_labels)
            loss_G.backward()
            optimizer_G.step()
        

        losses_D.append(loss_D.item())
        losses_G.append(loss_G.item())
        print(f"Epoch [{epoch+1}/{epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

    plt.figure(figsize=(10,5))
    plt.title("Generator and Discriminator Loss During Training")
    plt.plot(losses_G,label="G")
    plt.plot(losses_D,label="D")
    plt.xlabel("iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
    


## 6. Create and train a GAN

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

epochs = 50

train_gan(epochs, loader, D, G, optimizer_D, optimizer_G, criterion)

## 7. Generate and Visualize Fake Images

In [ ]:
def generate_fakes(G, n = 100, latent_size=64):
    z = torch.randn(n, latent_size).to(device)
    samples = G(z).detach().cpu()
    samples = samples.view(samples.size(0), 1, 28, 28)

    plt.figure(figsize=(6,6))
    for i in range(25):
        plt.subplot(5,5,i+1)
        plt.imshow(samples[i][0], cmap="gray")
        plt.axis("off")
    plt.suptitle("GAN Generated MNIST")
    plt.show()

generate_fakes(G)